# RM2 — Actor Type, Target, and Goals Typology
## Three Actor Types and Observed Community-Mass Exposure Mapping

Notebook ini melengkapi Rumusan Masalah 2 dengan tiga dimensi: **actor type**, **target**, dan **goals**.
Notebook ini hanya membaca output RM1 dan output sentimen RM2 yang sudah tersedia. Notebook ini **tidak**
menjalankan ulang Latent Coordination Network, Louvain, FSA_V/HCC, atau model transformer sentimen.

Output baru disimpan terpisah di `output/rm2_actor_type/`.

## 2. Kerangka Tiga Actor Type

Kerangka actor type yang ditetapkan dalam penelitian ini menggunakan tepat tiga kategori utama:

1. `Individual Actor` — Aktor Individual yang terverifikasi manual.
2. `Community Actor` — Aktor Komunitas Terkoordinasi, yaitu akun anggota HCC final.
3. `Mass Actor` — Aktor Massa/Audiens, yaitu komentator non-individual dan non-HCC.

Kolom utama adalah `actor_type_primary`; nilainya hanya boleh tiga kategori di atas.

## 3. Batasan Interpretasi

Mass Actor merupakan akun non-individual dan non-HCC yang berada dalam ruang percakapan TikTok yang dianalisis.
Hubungannya dengan Community Actor diukur sebagai paparan atau asosiasi yang teramati, bukan sebagai bukti
kausal bahwa opini massa telah dipengaruhi oleh komunitas.

Mass exposure score menggabungkan interaksi langsung, kedekatan temporal, dan kesamaan ruang video. Skor ini
mengukur intensitas paparan yang teramati, bukan besarnya pengaruh kausal.

Sentiment alignment menunjukkan keselarasan kategori sentimen yang teramati dan tidak membuktikan perpindahan
atau pengaruh opini.

## 4. Import Library dan Konfigurasi

In [1]:
import os
import re
import shutil
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

HEAD_COMMIT_BEFORE_IMPLEMENTATION = "4fcbdc93788f990e3eb51d2f25b57b09eb66ec27"

DATASET_PATH = Path("dataset.csv")
VIDEO_METADATA_PATH = Path("video_metadata_clean.csv")
LCN_NODES_PATH = Path("output/gephi/gephi_lcn_nodes.csv")
HCC_NODES_PATH = Path("output/gephi/gephi_hcc_nodes.csv")
HCC_EDGES_PATH = Path("output/gephi/gephi_hcc_edges.csv")
FOCAL_STRUCTURES_PATH = Path("output/tables/focal_structures.csv")
HCC_BRAND_PROFILE_PATH = Path("output/tables/hcc_brand_profile_auto.csv")
HCC_COSIM_PATH = Path("output/tables/hcc_cosimilarity_summary.csv")
COMMENT_SENTIMENT_PATH = Path("output/rm2_sentiment/tables/comment_sentiment.csv")
ACCOUNT_SENTIMENT_PATH = Path("output/rm2_sentiment/tables/account_sentiment_summary.csv")
HCC_GOALS_PATH = Path("output/rm2_sentiment/tables/hcc_sentiment_goals_summary.csv")
RM2_SENTIMENT_GEPHI_NODES_PATH = Path("output/rm2_sentiment/gephi/gephi_hcc_nodes_sentiment.csv")

OUT_DIR = Path("output/rm2_actor_type")
TABLE_DIR = OUT_DIR / "tables"
VIZ_DIR = OUT_DIR / "visualisasi"
GEPHI_DIR = OUT_DIR / "gephi"
CONFIG_DIR = Path("config")
REGISTRY_PATH = CONFIG_DIR / "individual_actor_registry.csv"
for d in [TABLE_DIR, VIZ_DIR, GEPHI_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ACTOR_TYPES = ["Individual Actor", "Community Actor", "Mass Actor"]
MASS_TEMPORAL_WINDOW_MINUTES = 60
CENTRALITY_Q25 = 0.25
CENTRALITY_Q75 = 0.75
ACCOUNT_GOAL_MIN_COMMENTS = 3
GOAL_POSITIVE_THRESHOLD = 0.60
GOAL_NEGATIVE_THRESHOLD = 0.60
GOAL_NEUTRAL_THRESHOLD = 0.60
GOAL_POLARIZED_THRESHOLD = 0.40
EXPOSURE_WEIGHTS = {"direct": 3, "temporal": 2, "shared": 1}
EXPOSURE_ORDER = [
    "Direct Interaction", "Temporal-Context Exposure",
    "Shared-Video Exposure", "No Observed HCC Exposure",
]
SENTIMENT_ORDER = ["Positive", "Neutral", "Negative"]

print(f"Commit HEAD sebelum implementasi: {HEAD_COMMIT_BEFORE_IMPLEMENTATION}")
print(f"Output baru: {OUT_DIR}/")

Commit HEAD sebelum implementasi: 4fcbdc93788f990e3eb51d2f25b57b09eb66ec27
Output baru: output\rm2_actor_type/


## 5. Audit File Input

In [2]:
ID_DTYPES = {
    "comment_id": str, "parent_comment_id": str, "video_id": str, "user_id": str,
    "username": str, "parent_user": str, "id": str, "label": str,
    "source": str, "target": str,
}

INPUT_FILES = {
    "dataset": (DATASET_PATH, False),
    "video_metadata": (VIDEO_METADATA_PATH, True),
    "lcn_nodes": (LCN_NODES_PATH, True),
    "hcc_nodes": (HCC_NODES_PATH, False),
    "hcc_edges": (HCC_EDGES_PATH, False),
    "focal_structures": (FOCAL_STRUCTURES_PATH, True),
    "hcc_brand_profile": (HCC_BRAND_PROFILE_PATH, True),
    "hcc_cosimilarity_summary": (HCC_COSIM_PATH, True),
    "comment_sentiment": (COMMENT_SENTIMENT_PATH, False),
    "account_sentiment_summary": (ACCOUNT_SENTIMENT_PATH, False),
    "hcc_sentiment_goals_summary": (HCC_GOALS_PATH, False),
    "rm2_sentiment_gephi_hcc_nodes": (RM2_SENTIMENT_GEPHI_NODES_PATH, True),
}

def dtype_for(path):
    cols = pd.read_csv(path, nrows=0).columns
    return {k: v for k, v in ID_DTYPES.items() if k in cols}

audit_rows = []
missing_required = []
for label, (path, optional) in INPUT_FILES.items():
    if not path.exists():
        status = "MISSING_OPTIONAL" if optional else "MISSING_REQUIRED"
        if not optional:
            missing_required.append(str(path))
        audit_rows.append({"label": label, "path": str(path), "optional": optional,
                           "exists": False, "n_rows": 0, "n_columns": 0,
                           "columns": [], "status": status})
        print(f"WARNING: {status}: {path}")
        continue
    df0 = pd.read_csv(path, low_memory=False, dtype=dtype_for(path))
    audit_rows.append({"label": label, "path": str(path), "optional": optional,
                       "exists": True, "n_rows": df0.shape[0], "n_columns": df0.shape[1],
                       "columns": list(df0.columns), "status": "OK"})
if missing_required:
    raise FileNotFoundError(missing_required)

input_audit = pd.DataFrame(audit_rows)
for r in audit_rows:
    print(f"- {r['label']}: {r['status']} | shape=({r['n_rows']}, {r['n_columns']})")
    print(f"  columns={r['columns']}")
input_audit

- dataset: OK | shape=(33847, 18)
  columns=['product_category', 'video_url', 'video_id', 'comment_type', 'comment_id', 'parent_comment_id', 'parent_user', 'reply_count', 'expected_reply_count', 'actual_reply_scraped', 'thread_level', 'user_id', 'username', 'text', 'mentions', 'hashtags', 'timestamp', 'source_file']
- video_metadata: OK | shape=(55, 14)
  columns=['no', 'product_category', 'video_url', 'video_id', 'creator', 'caption', 'hashtags', 'mentions', 'view_count', 'like_count', 'comment_count', 'upload_date', 'n_hashtags', 'n_mentions']
- lcn_nodes: OK | shape=(724, 7)
  columns=['id', 'label', 'degree', 'weighted_degree', 'betweenness', 'community', 'brand_label_auto']
- hcc_nodes: OK | shape=(218, 11)
  columns=['id', 'label', 'degree', 'weighted_degree', 'betweenness', 'community', 'primary_brand', 'brand_label_auto', 'brand_combo', 'brand_confidence', 'narrative_similarity_level']
- hcc_edges: OK | shape=(464, 9)
  columns=['source', 'target', 'weight', 'n_evidence', 'co_c

,label,path,optional,exists,n_rows,n_columns,columns,status
0,dataset,dataset.csv,False,True,33847,18,"[product_category, video_url, video_id, commen...",OK
1,video_metadata,video_metadata_clean.csv,True,True,55,14,"[no, product_category, video_url, video_id, cr...",OK
2,lcn_nodes,output\gephi\gephi_lcn_nodes.csv,True,True,724,7,"[id, label, degree, weighted_degree, betweenne...",OK
3,hcc_nodes,output\gephi\gephi_hcc_nodes.csv,False,True,218,11,"[id, label, degree, weighted_degree, betweenne...",OK
4,hcc_edges,output\gephi\gephi_hcc_edges.csv,False,True,464,9,"[source, target, weight, n_evidence, co_conv, ...",OK
5,focal_structures,output\tables\focal_structures.csv,True,True,207,11,"[Community, Nodes (Louvain), Edges (Louvain), ...",OK
6,hcc_brand_profile,output\tables\hcc_brand_profile_auto.csv,True,True,42,25,"[hcc_id, community_size, n_unique_hashtags, pr...",OK
7,hcc_cosimilarity_summary,output\tables\hcc_cosimilarity_summary.csv,True,True,42,18,"[hcc_id, brand_label_auto, primary_brand, bran...",OK
8,comment_sentiment,output\rm2_sentiment\tables\comment_sentiment.csv,False,True,33847,20,"[comment_id, username, user_id, video_id, prod...",OK
9,account_sentiment_summary,output\rm2_sentiment\tables\account_sentiment_...,False,True,26424,21,"[username, n_comments, n_valid_text_comments, ...",OK


## 6. Normalisasi Username

In [3]:
def normalize_username(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    if text.startswith("@"):
        text = text[1:]
    return text

def norm_id(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return "" if text.lower() in {"", "nan", "none"} else text

def read_csv(path, optional=False):
    if not path.exists():
        if optional:
            return pd.DataFrame()
        raise FileNotFoundError(path)
    return pd.read_csv(path, low_memory=False, dtype=dtype_for(path))

def brand_from_text(value):
    if pd.isna(value):
        return "Not identified"
    text = str(value).lower()
    if "originote" in text:
        return "The Originote"
    if "azarine" in text:
        return "Azarine"
    if "daviena" in text or "davina" in text:
        return "Daviena"
    if "maryame" in text:
        return "Maryame"
    return "Not identified"

def target_scope(label):
    label = "" if pd.isna(label) else str(label).strip()
    if label in {"", "nan", "None", "Non-HCC", "Not identified"}:
        return "Unidentified Brand Context"
    if "Mixed" in label:
        return "Cross-brand Context"
    return "Single-brand Context"

def hcc_key(value):
    try:
        return str(int(float(value)))
    except Exception:
        return "" if pd.isna(value) else str(value)

def hcc_sort(value):
    try:
        return (0, int(float(value)))
    except Exception:
        return (1, str(value))

dataset = read_csv(DATASET_PATH)
video_metadata = read_csv(VIDEO_METADATA_PATH, optional=True)
lcn_nodes = read_csv(LCN_NODES_PATH, optional=True)
hcc_nodes = read_csv(HCC_NODES_PATH)
hcc_edges = read_csv(HCC_EDGES_PATH)
focal_structures = read_csv(FOCAL_STRUCTURES_PATH, optional=True)
hcc_brand_profile = read_csv(HCC_BRAND_PROFILE_PATH, optional=True)
hcc_cosim = read_csv(HCC_COSIM_PATH, optional=True)
comment_sentiment = read_csv(COMMENT_SENTIMENT_PATH)
account_sentiment = read_csv(ACCOUNT_SENTIMENT_PATH)
hcc_goals = read_csv(HCC_GOALS_PATH)
gephi_sentiment_nodes = read_csv(RM2_SENTIMENT_GEPHI_NODES_PATH, optional=True)

for df, cols in [
    (dataset, ["username", "parent_user", "video_id", "comment_id", "parent_comment_id"]),
    (comment_sentiment, ["username", "parent_user", "video_id", "comment_id", "parent_comment_id"]),
    (account_sentiment, ["username"]),
    (hcc_nodes, ["id"]),
    (lcn_nodes, ["id"] if not lcn_nodes.empty else []),
]:
    for c in cols:
        if c in df.columns:
            df[c + "_norm"] = df[c].apply(normalize_username if c in {"username", "parent_user", "id"} else norm_id)

comment_sentiment["timestamp_dt"] = pd.to_datetime(comment_sentiment["timestamp"], errors="coerce", utc=True)
hcc_missing_dataset = sorted(set(hcc_nodes["id_norm"]) - set(dataset["username_norm"]))
print(f"Username unik dataset: {dataset['username_norm'].nunique():,}")
print(f"Akun HCC: {hcc_nodes['id_norm'].nunique():,}")
print(f"Akun LCN: {lcn_nodes['id_norm'].nunique() if not lcn_nodes.empty else 0:,}")
print(f"Merge HCC ke dataset: {(1 - len(hcc_missing_dataset) / max(1, hcc_nodes['id_norm'].nunique())) * 100:.2f}%")
print(f"Duplicate username setelah normalisasi: {dataset.loc[dataset['username_norm'].duplicated(), 'username_norm'].nunique():,}")
print(f"Akun HCC tidak ditemukan di dataset: {len(hcc_missing_dataset):,}")
print(f"Timestamp parsed: {comment_sentiment['timestamp_dt'].notna().sum():,} / {len(comment_sentiment):,}")

Username unik dataset: 26,424
Akun HCC: 218
Akun LCN: 724
Merge HCC ke dataset: 100.00%
Duplicate username setelah normalisasi: 3,165
Akun HCC tidak ditemukan di dataset: 0
Timestamp parsed: 33,788 / 33,847


## 7. Load Individual Actor Registry

In [4]:
REGISTRY_COLUMNS = [
    "username", "display_name", "individual_subtype", "associated_brand",
    "identification_source", "verification_status", "notes",
]
ALLOWED_SUBTYPES = {
    "Official Brand Account", "Influencer", "Video Creator",
    "Owner / Business Representative", "Expert / Professional",
    "Other Individual Source",
}
ALLOWED_STATUS = {"Verified", "Candidate", "Rejected"}

if not REGISTRY_PATH.exists():
    pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(REGISTRY_PATH, index=False)
    print(f"WARNING: registry dibuat dengan header saja: {REGISTRY_PATH}")
    print("WARNING: Individual Actor memerlukan verifikasi manual; jumlahnya dapat nol.")

registry = pd.read_csv(REGISTRY_PATH, dtype=str).fillna("")
for c in REGISTRY_COLUMNS:
    if c not in registry.columns:
        registry[c] = ""
registry = registry[REGISTRY_COLUMNS].copy()
registry["username_norm"] = registry["username"].apply(normalize_username)
registry["verification_status"] = registry["verification_status"].str.strip()
registry["individual_subtype"] = registry["individual_subtype"].str.strip()

bad_status = sorted(set(registry.loc[registry["verification_status"].ne("") & ~registry["verification_status"].isin(ALLOWED_STATUS), "verification_status"]))
bad_subtype = sorted(set(registry.loc[registry["individual_subtype"].ne("") & ~registry["individual_subtype"].isin(ALLOWED_SUBTYPES), "individual_subtype"]))
if bad_status:
    print(f"WARNING: status registry tidak dikenal: {bad_status}")
if bad_subtype:
    print(f"WARNING: subtype registry tidak dikenal: {bad_subtype}")

verified_registry = registry[(registry["verification_status"] == "Verified") & registry["username_norm"].ne("")].drop_duplicates("username_norm")
candidate_registry = registry[(registry["verification_status"] == "Candidate") & registry["username_norm"].ne("")]
print(f"Registry rows: {len(registry):,}")
print(f"Verified Individual Actor: {len(verified_registry):,}")
print(f"Candidate registry: {len(candidate_registry):,}")

Registry rows: 0
Verified Individual Actor: 0
Candidate registry: 0


## 8. Generate Individual Actor Candidates

In [5]:
CREATOR_COLS = ["creator_username", "author_username", "video_author", "author", "username_creator", "account_name", "creator"]
candidate_frames = []
if not video_metadata.empty:
    detected = [c for c in video_metadata.columns if c.lower() in {x.lower() for x in CREATOR_COLS}]
    print(f"Kolom kreator terdeteksi: {detected if detected else 'Tidak ada'}")
    for c in detected:
        tmp = video_metadata.copy()
        tmp["username"] = tmp[c].apply(normalize_username)
        tmp = tmp[tmp["username"].ne("")]
        if tmp.empty:
            continue
        tmp["brand_item"] = tmp["product_category"].apply(brand_from_text) if "product_category" in tmp.columns else "Not identified"
        g = tmp.groupby("username", as_index=False).agg(
            n_videos=(c, "size"),
            possible_brand=("brand_item", lambda s: s.value_counts().index[0] if len(s) else "Not identified"),
        )
        g["candidate_source"] = f"{VIDEO_METADATA_PATH.name}:{c}"
        g["suggested_subtype"] = "Video Creator" if c.lower() in {"creator", "creator_username", "author", "author_username", "video_author"} else "Other Individual Source"
        candidate_frames.append(g)

if candidate_frames:
    individual_actor_candidates = pd.concat(candidate_frames, ignore_index=True).drop_duplicates(["username", "candidate_source"])
else:
    individual_actor_candidates = pd.DataFrame(columns=["username", "n_videos", "possible_brand", "candidate_source", "suggested_subtype"])
individual_actor_candidates["verification_status"] = "Candidate"
individual_actor_candidates["notes"] = ""
individual_actor_candidates = individual_actor_candidates[["username", "candidate_source", "n_videos", "possible_brand", "suggested_subtype", "verification_status", "notes"]]
individual_actor_candidates.to_csv(TABLE_DIR / "individual_actor_candidates.csv", index=False)
print(f"Disimpan: {TABLE_DIR / 'individual_actor_candidates.csv'} ({len(individual_actor_candidates):,} kandidat)")
individual_actor_candidates.head()

Kolom kreator terdeteksi: ['creator']
Disimpan: output\rm2_actor_type\tables\individual_actor_candidates.csv (43 kandidat)


,username,candidate_source,n_videos,possible_brand,suggested_subtype,verification_status,notes
0,adilana.na,video_metadata_clean.csv:creator,1,Daviena,Video Creator,Candidate,
1,adiliarizqa,video_metadata_clean.csv:creator,1,Daviena,Video Creator,Candidate,
2,afikarrhnur,video_metadata_clean.csv:creator,1,Azarine,Video Creator,Candidate,
3,ansyashine,video_metadata_clean.csv:creator,1,Azarine,Video Creator,Candidate,
4,asabelavvv,video_metadata_clean.csv:creator,1,Azarine,Video Creator,Candidate,


## 9. Identifikasi HCC dan LCN Membership

In [6]:
accounts = account_sentiment.copy()
accounts["username_norm"] = accounts["username"].apply(normalize_username)
accounts = accounts.sort_values("username_norm").drop_duplicates("username_norm")

hcc_lookup = hcc_nodes.drop_duplicates("id_norm").set_index("id_norm")
lcn_lookup = lcn_nodes.drop_duplicates("id_norm").set_index("id_norm") if not lcn_nodes.empty else pd.DataFrame()
verified_set = set(verified_registry["username_norm"])

accounts["is_individual_actor"] = accounts["username_norm"].isin(verified_set)
accounts["is_hcc_member"] = accounts["username_norm"].isin(hcc_lookup.index)
accounts["is_lcn_member"] = accounts["username_norm"].isin(lcn_lookup.index) if not lcn_lookup.empty else False
accounts["individual_in_hcc"] = accounts["is_individual_actor"] & accounts["is_hcc_member"]

hcc_cols = ["id_norm", "community", "degree", "weighted_degree", "betweenness", "primary_brand", "brand_label_auto", "brand_combo", "brand_confidence", "narrative_similarity_level"]
accounts = accounts.merge(hcc_nodes[hcc_cols], left_on="username_norm", right_on="id_norm", how="left", suffixes=("", "_hcc_node")).drop(columns=["id_norm"])
if not lcn_lookup.empty:
    lcn_cols = ["id_norm"] + [c for c in ["degree", "weighted_degree", "betweenness", "community", "brand_label_auto"] if c in lcn_nodes.columns]
    lcn_tmp = lcn_nodes[lcn_cols].rename(columns={"degree": "lcn_degree", "weighted_degree": "lcn_weighted_degree", "betweenness": "lcn_betweenness", "community": "lcn_community", "brand_label_auto": "lcn_brand_label_auto"})
    accounts = accounts.merge(lcn_tmp, left_on="username_norm", right_on="id_norm", how="left").drop(columns=["id_norm"])
else:
    for c in ["lcn_degree", "lcn_weighted_degree", "lcn_betweenness", "lcn_community", "lcn_brand_label_auto"]:
        accounts[c] = np.nan

if not verified_registry.empty:
    accounts = accounts.merge(verified_registry[["username_norm", "individual_subtype", "associated_brand", "identification_source", "verification_status", "display_name", "notes"]], on="username_norm", how="left")
else:
    for c in ["individual_subtype", "associated_brand", "identification_source", "verification_status", "display_name", "notes"]:
        accounts[c] = ""

print(f"Total akun unik: {len(accounts):,}")
print(f"HCC member: {accounts['is_hcc_member'].sum():,}")
print(f"LCN member: {accounts['is_lcn_member'].sum():,}")
print(f"Verified Individual Actor: {accounts['is_individual_actor'].sum():,}")
print(f"Duplicate rows setelah merge: {accounts['username_norm'].duplicated().sum():,}")

Total akun unik: 26,424
HCC member: 218
LCN member: 724
Verified Individual Actor: 0
Duplicate rows setelah merge: 0


## 10. Klasifikasi `actor_type_primary`

In [7]:
accounts["actor_type_primary"] = np.select(
    [accounts["is_individual_actor"], accounts["is_hcc_member"]],
    ["Individual Actor", "Community Actor"],
    default="Mass Actor",
)
accounts["is_mass_actor"] = accounts["actor_type_primary"].eq("Mass Actor")
assert set(accounts["actor_type_primary"].unique()).issubset(set(ACTOR_TYPES))
assert len(accounts) == accounts["username_norm"].nunique()
print(accounts["actor_type_primary"].value_counts().reindex(ACTOR_TYPES, fill_value=0).to_string())

actor_type_primary
Individual Actor        0
Community Actor       218
Mass Actor          26206


## 11. Karakteristik Community Actor

In [8]:
def centrality_series(g):
    values = pd.to_numeric(g["weighted_degree"], errors="coerce")
    if len(g) < 3 or values.nunique(dropna=True) <= 1:
        return pd.Series("Uniform / Insufficient variation", index=g.index)
    q25, q75 = values.quantile(CENTRALITY_Q25), values.quantile(CENTRALITY_Q75)
    if np.isclose(q25, q75):
        return pd.Series("Uniform / Insufficient variation", index=g.index)
    return values.apply(lambda v: "High" if v >= q75 else ("Low" if v <= q25 else "Medium"))

hcc_cent = hcc_nodes[["id_norm", "community"]].copy()
hcc_cent["community_centrality_level"] = hcc_nodes.groupby("community", group_keys=False).apply(centrality_series).reindex(hcc_nodes.index).values
hcc_cent["community_network_descriptor"] = hcc_cent["community_centrality_level"].map({
    "High": "Structurally central", "Medium": "Intermediate", "Low": "Peripheral",
    "Uniform / Insufficient variation": "Uniform / Insufficient variation",
})
accounts = accounts.merge(hcc_cent[["id_norm", "community_centrality_level", "community_network_descriptor"]], left_on="username_norm", right_on="id_norm", how="left").drop(columns=["id_norm"])
accounts[["community_centrality_level", "community_network_descriptor"]] = accounts[["community_centrality_level", "community_network_descriptor"]].fillna("")
print(accounts.loc[accounts["is_hcc_member"], "community_centrality_level"].value_counts().to_string())

community_centrality_level
Uniform / Insufficient variation    130
Low                                  44
High                                 32
Medium                               12

## 12. Identifikasi Mass Actor

In [9]:
accounts["mass_network_position"] = np.where(accounts["is_mass_actor"], np.where(accounts["is_lcn_member"], "LCN Non-HCC", "Outside LCN"), "")
mass_accounts = accounts.loc[accounts["is_mass_actor"], ["username", "username_norm", "mass_network_position"]].copy()
print(f"Mass Actor: {len(mass_accounts):,}")
print(mass_accounts["mass_network_position"].value_counts().to_string())

Mass Actor: 26,206
mass_network_position
Outside LCN    25700
LCN Non-HCC      506


## 13. Community-Mass Observed Exposure

In [10]:
comments = comment_sentiment.copy()
comments["username_norm"] = comments["username"].apply(normalize_username)
comments["parent_user_norm"] = comments["parent_user"].apply(normalize_username)
comments["video_id_norm"] = comments["video_id"].apply(norm_id)
comments["comment_id_norm"] = comments["comment_id"].apply(norm_id)
comments["parent_comment_id_norm"] = comments["parent_comment_id"].apply(norm_id)
comments["timestamp_dt"] = pd.to_datetime(comments["timestamp"], errors="coerce", utc=True)
comments["hcc_id"] = comments["username_norm"].map(hcc_lookup["community"])
comments["parent_hcc_id"] = comments["parent_user_norm"].map(hcc_lookup["community"])

mass_set = set(mass_accounts["username_norm"])
hcc_set = set(hcc_lookup.index)
assoc = defaultdict(lambda: {"shared_videos": set(), "shared_hcc_accounts": set(), "mass_to_hcc": 0, "hcc_to_mass": 0, "temporal": 0, "minutes": []})
mass_video_any_hcc = defaultdict(set)
mass_video_hcc_users = defaultdict(set)

for video_id, g in comments.groupby("video_id_norm", sort=False):
    if video_id == "":
        continue
    hg = g[g["username_norm"].isin(hcc_set) & g["hcc_id"].notna()]
    mg = g[g["username_norm"].isin(mass_set)]
    if hg.empty or mg.empty:
        continue
    for hcc_id, hh in hg.groupby("hcc_id"):
        hcc_id_s = hcc_key(hcc_id)
        hcc_users = set(hh["username_norm"])
        for mass_user in mg["username_norm"].dropna().unique():
            assoc[(mass_user, hcc_id_s)]["shared_videos"].add(video_id)
            assoc[(mass_user, hcc_id_s)]["shared_hcc_accounts"].update(hcc_users)
            mass_video_any_hcc[mass_user].add(video_id)
            mass_video_hcc_users[mass_user].update(hcc_users)

parent_valid = comments["parent_user_norm"].ne("")
for _, r in comments[comments["username_norm"].isin(mass_set) & parent_valid & comments["parent_user_norm"].isin(hcc_set)].iterrows():
    assoc[(r["username_norm"], hcc_key(r["parent_hcc_id"]))]["mass_to_hcc"] += 1
for _, r in comments[comments["username_norm"].isin(hcc_set) & parent_valid & comments["parent_user_norm"].isin(mass_set)].iterrows():
    assoc[(r["parent_user_norm"], hcc_key(r["hcc_id"]))]["hcc_to_mass"] += 1

temporal_records = []
window = pd.Timedelta(minutes=MASS_TEMPORAL_WINDOW_MINUTES)
for video_id, g in comments[comments["timestamp_dt"].notna()].groupby("video_id_norm", sort=False):
    hg = g[g["username_norm"].isin(hcc_set) & g["hcc_id"].notna()][["hcc_id", "timestamp_dt"]].sort_values("timestamp_dt")
    mg = g[g["username_norm"].isin(mass_set)][["username_norm", "timestamp_dt"]].sort_values("timestamp_dt")
    if hg.empty or mg.empty:
        continue
    for _, m in mg.iterrows():
        prior = hg[(hg["timestamp_dt"] <= m["timestamp_dt"]) & (hg["timestamp_dt"] >= m["timestamp_dt"] - window)]
        if prior.empty:
            continue
        for hcc_id, hh in prior.groupby("hcc_id"):
            diffs = ((m["timestamp_dt"] - hh["timestamp_dt"]).dt.total_seconds() / 60)
            diffs = diffs[(diffs >= 0) & (diffs <= MASS_TEMPORAL_WINDOW_MINUTES)]
            if diffs.empty:
                continue
            key = (m["username_norm"], hcc_key(hcc_id))
            assoc[key]["temporal"] += int(len(diffs))
            assoc[key]["minutes"].extend([float(x) for x in diffs])
            temporal_records.extend([{"mass_username": m["username_norm"], "hcc_id": hcc_key(hcc_id), "minutes_after_hcc": float(x)} for x in diffs])

assoc_rows = []
for (mass_user, hcc_id), v in assoc.items():
    direct = v["mass_to_hcc"] + v["hcc_to_mass"]
    shared = len(v["shared_videos"])
    temporal = v["temporal"]
    if direct == shared == temporal == 0:
        continue
    assoc_rows.append({
        "mass_username": mass_user, "hcc_id": hcc_id, "shared_video_count": shared,
        "direct_reply_count": direct, "temporal_after_count": temporal,
        "min_minutes_after_hcc": np.nanmin(v["minutes"]) if v["minutes"] else np.nan,
        "association_score": EXPOSURE_WEIGHTS["direct"] * direct + EXPOSURE_WEIGHTS["temporal"] * temporal + shared,
        "is_primary_association": False,
        "mass_replies_to_hcc_count": v["mass_to_hcc"], "hcc_replies_to_mass_count": v["hcc_to_mass"],
        "median_minutes_after_hcc": np.nanmedian(v["minutes"]) if v["minutes"] else np.nan,
    })
mass_hcc_association = pd.DataFrame(assoc_rows)
if mass_hcc_association.empty:
    mass_hcc_association = pd.DataFrame(columns=["mass_username", "hcc_id", "shared_video_count", "direct_reply_count", "temporal_after_count", "min_minutes_after_hcc", "association_score", "is_primary_association", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count", "median_minutes_after_hcc"])

mass_exposure = mass_accounts.copy()
for c in ["shared_hcc_video_count", "n_hcc_accounts_shared_video", "direct_reply_with_hcc_count", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count", "temporal_after_hcc_count", "associated_hcc_count", "mass_exposure_score"]:
    mass_exposure[c] = 0
for c in ["min_minutes_after_hcc", "median_minutes_after_hcc"]:
    mass_exposure[c] = np.nan
for c in ["associated_hcc_ids", "primary_associated_hcc", "association_basis", "mass_exposure_level"]:
    mass_exposure[c] = ""

if not mass_hcc_association.empty:
    summed = mass_hcc_association.groupby("mass_username")[["direct_reply_count", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count", "temporal_after_count"]].sum()
    ids = mass_hcc_association.groupby("mass_username")["hcc_id"].apply(lambda s: ";".join(sorted(set(map(str, s)), key=hcc_sort)))
    min_m = mass_hcc_association.groupby("mass_username")["min_minutes_after_hcc"].min()
    med_m = pd.DataFrame(temporal_records).groupby("mass_username")["minutes_after_hcc"].median() if temporal_records else pd.Series(dtype=float)
else:
    summed = pd.DataFrame()
    ids = pd.Series(dtype=str)
    min_m = pd.Series(dtype=float)
    med_m = pd.Series(dtype=float)

mass_exposure = mass_exposure.set_index("username_norm")
for u in mass_exposure.index:
    mass_exposure.loc[u, "shared_hcc_video_count"] = len(mass_video_any_hcc.get(u, set()))
    mass_exposure.loc[u, "n_hcc_accounts_shared_video"] = len(mass_video_hcc_users.get(u, set()))
    if u in summed.index:
        mass_exposure.loc[u, "direct_reply_with_hcc_count"] = int(summed.loc[u, "direct_reply_count"])
        mass_exposure.loc[u, "mass_replies_to_hcc_count"] = int(summed.loc[u, "mass_replies_to_hcc_count"])
        mass_exposure.loc[u, "hcc_replies_to_mass_count"] = int(summed.loc[u, "hcc_replies_to_mass_count"])
        mass_exposure.loc[u, "temporal_after_hcc_count"] = int(summed.loc[u, "temporal_after_count"])
    if u in ids.index:
        mass_exposure.loc[u, "associated_hcc_ids"] = ids.loc[u]
        mass_exposure.loc[u, "associated_hcc_count"] = len(ids.loc[u].split(";")) if ids.loc[u] else 0
    if u in min_m.index:
        mass_exposure.loc[u, "min_minutes_after_hcc"] = min_m.loc[u]
    if u in med_m.index:
        mass_exposure.loc[u, "median_minutes_after_hcc"] = med_m.loc[u]
mass_exposure = mass_exposure.reset_index()
mass_exposure["mass_exposure_score"] = 3 * mass_exposure["direct_reply_with_hcc_count"] + 2 * mass_exposure["temporal_after_hcc_count"] + mass_exposure["shared_hcc_video_count"]
print(f"Mass-HCC association rows: {len(mass_hcc_association):,}")

Mass-HCC association rows: 107,318


## 14. Primary HCC Association

In [11]:
if not mass_hcc_association.empty:
    tmp = mass_hcc_association.copy()
    tmp["_hcc_sort"] = tmp["hcc_id"].apply(hcc_sort)
    primary = tmp.sort_values(["mass_username", "direct_reply_count", "temporal_after_count", "shared_video_count", "_hcc_sort"], ascending=[True, False, False, False, True]).drop_duplicates("mass_username")
    primary_keys = set(zip(primary["mass_username"], primary["hcc_id"]))
    mass_hcc_association["is_primary_association"] = [(m, h) in primary_keys for m, h in zip(mass_hcc_association["mass_username"], mass_hcc_association["hcc_id"])]
    pmap = primary.set_index("mass_username")
else:
    pmap = pd.DataFrame()

mass_exposure = mass_exposure.set_index("username_norm")
for u, r in pmap.iterrows():
    mass_exposure.loc[u, "primary_associated_hcc"] = r["hcc_id"]
    mass_exposure.loc[u, "association_basis"] = (
        "Direct reply with HCC" if r["direct_reply_count"] > 0 else
        "Temporal association after HCC comment" if r["temporal_after_count"] > 0 else
        "Shared video context" if r["shared_video_count"] > 0 else
        "No observed HCC exposure"
    )
mass_exposure["mass_exposure_level"] = np.select(
    [
        mass_exposure["direct_reply_with_hcc_count"] > 0,
        mass_exposure["temporal_after_hcc_count"] > 0,
        mass_exposure["shared_hcc_video_count"] > 0,
    ],
    ["Direct Interaction", "Temporal-Context Exposure", "Shared-Video Exposure"],
    default="No Observed HCC Exposure",
)
mass_exposure["association_basis"] = mass_exposure["association_basis"].replace("", "No observed HCC exposure")
mass_exposure = mass_exposure.reset_index()
mass_hcc_association = mass_hcc_association[["mass_username", "hcc_id", "shared_video_count", "direct_reply_count", "temporal_after_count", "min_minutes_after_hcc", "association_score", "is_primary_association", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count", "median_minutes_after_hcc"]]
mass_hcc_association.to_csv(TABLE_DIR / "mass_hcc_association.csv", index=False)
print(mass_exposure["mass_exposure_level"].value_counts().reindex(EXPOSURE_ORDER, fill_value=0).to_string())

mass_exposure_level
Direct Interaction              28
Temporal-Context Exposure     2294
Shared-Video Exposure        23884
No Observed HCC Exposure         0


## 15. Target Mapping

In [12]:
comments["observed_brand"] = comments["product_category"].apply(brand_from_text)
comments["thread_key"] = np.where(comments["parent_comment_id_norm"].ne(""), comments["parent_comment_id_norm"], comments["comment_id_norm"])
target_rows = []
for u, g in comments.groupby("username_norm", sort=False):
    b = g["observed_brand"].replace("Not identified", np.nan).dropna().value_counts()
    if len(b):
        primary_brand = b.index[0]
        label = primary_brand if len(b) == 1 else ("Mixed_2_Brands" if len(b) == 2 else "Mixed_3plus_Brands")
        share = b.iloc[0] / b.sum()
        conf = "High" if share >= 0.60 else ("Medium" if share >= 0.40 else "Low")
    else:
        primary_brand, label, conf = "Not identified", "Not identified", "None"
    pc = g["product_category"].dropna().astype(str).value_counts()
    target_rows.append({
        "username_norm": u, "mass_observed_target_brand": primary_brand, "comment_target_brand_label": label,
        "target_product_category": pc.index[0] if len(pc) else "Not identified",
        "target_video_count": g["video_id_norm"].nunique(), "target_thread_count": g["thread_key"].replace("", np.nan).nunique(),
        "target_confidence_from_comments": conf,
    })
accounts = accounts.merge(pd.DataFrame(target_rows), on="username_norm", how="left")

hcc_brand_map = hcc_goals[["hcc_id", "primary_brand", "brand_label_auto", "brand_confidence"]].copy()
hcc_brand_map["hcc_id_str"] = hcc_brand_map["hcc_id"].astype(str)
hcc_brand_map = hcc_brand_map.set_index("hcc_id_str")
mass_exposure["primary_associated_hcc"] = mass_exposure["primary_associated_hcc"].fillna("").astype(str)
mass_exposure["associated_hcc_brand_context"] = mass_exposure["primary_associated_hcc"].map(hcc_brand_map["primary_brand"]).fillna("Not identified")

accounts = accounts.merge(mass_exposure[["username_norm", "primary_associated_hcc", "associated_hcc_brand_context", "mass_exposure_level", "mass_exposure_score"]], on="username_norm", how="left")

def resolve_target(r):
    if r["actor_type_primary"] == "Community Actor":
        primary = r["primary_brand"] if pd.notna(r["primary_brand"]) else "Not identified"
        label = r["brand_label_auto"] if pd.notna(r["brand_label_auto"]) else "Not identified"
        conf, source = r["brand_confidence"], "HCC brand profile / RM1 context"
    elif r["actor_type_primary"] == "Individual Actor":
        primary = r["associated_brand"] if pd.notna(r["associated_brand"]) and r["associated_brand"] != "" else "Not identified"
        label = primary
        conf, source = ("Verified registry" if primary != "Not identified" else "None"), "verified individual registry"
    else:
        primary, label = r["mass_observed_target_brand"], r["comment_target_brand_label"]
        conf, source = r["target_confidence_from_comments"], "commented-video product_category; associated HCC retained separately"
    primary = primary if str(primary) not in {"", "nan", "Non-HCC"} else "Not identified"
    label = label if str(label) not in {"", "nan", "Non-HCC"} else "Not identified"
    return pd.Series({"target_brand_primary": primary, "target_brand_label": label, "target_scope": target_scope(label), "target_confidence": conf, "target_source": source})

accounts = pd.concat([accounts, accounts.apply(resolve_target, axis=1)], axis=1)
accounts["target_product_category"] = accounts["target_product_category"].fillna("Not identified")
accounts["target_video_count"] = accounts["target_video_count"].fillna(0).astype(int)
accounts["target_thread_count"] = accounts["target_thread_count"].fillna(0).astype(int)
print(accounts["target_scope"].value_counts().to_string())

target_scope
Single-brand Context          25838
Cross-brand Context             580
Unidentified Brand Context        6


## 16. Integrasi Goals dari Notebook Sentimen

In [13]:
def goal_orientation(pos, neu, neg, n):
    if pd.isna(n) or n < ACCOUNT_GOAL_MIN_COMMENTS:
        return "Insufficient Text"
    if pos >= GOAL_POSITIVE_THRESHOLD:
        return "Promotional / Supportive"
    if neg >= GOAL_NEGATIVE_THRESHOLD:
        return "Critical / Complaint"
    if neu >= GOAL_NEUTRAL_THRESHOLD:
        return "Neutral Engagement"
    if pos >= GOAL_POLARIZED_THRESHOLD and neg >= GOAL_POLARIZED_THRESHOLD:
        return "Polarized / Contested"
    return "Mixed Goals"

def goal_conf(pos, neu, neg, n, orient):
    if orient == "Insufficient Text":
        return "None"
    d = max(float(pos or 0), float(neu or 0), float(neg or 0))
    return "High" if d >= 0.75 and n >= 10 else ("Medium" if d >= 0.60 and n >= ACCOUNT_GOAL_MIN_COMMENTS else "Low")

accounts["account_goal_orientation"] = accounts.apply(lambda r: goal_orientation(r["positive_ratio"], r["neutral_ratio"], r["negative_ratio"], r["n_valid_text_comments"]), axis=1)
accounts["account_goal_confidence"] = accounts.apply(lambda r: goal_conf(r["positive_ratio"], r["neutral_ratio"], r["negative_ratio"], r["n_valid_text_comments"], r["account_goal_orientation"]), axis=1)

hcc_goal_cols = ["hcc_id", "dominant_sentiment", "positive_ratio", "neutral_ratio", "negative_ratio", "goal_orientation", "goal_confidence", "avg_sentiment_confidence", "n_valid_text_comments", "n_comments"]
hcc_tmp = hcc_goals[hcc_goal_cols].rename(columns={
    "hcc_id": "hcc_id_goal", "dominant_sentiment": "hcc_dominant_sentiment",
    "positive_ratio": "hcc_positive_ratio", "neutral_ratio": "hcc_neutral_ratio",
    "negative_ratio": "hcc_negative_ratio", "goal_orientation": "hcc_goal_orientation",
    "goal_confidence": "hcc_goal_confidence", "avg_sentiment_confidence": "hcc_avg_sentiment_confidence",
    "n_valid_text_comments": "hcc_n_valid_text_comments", "n_comments": "hcc_n_comments",
})
hcc_tmp["hcc_id_goal_str"] = hcc_tmp["hcc_id_goal"].astype(str)
accounts["community_str"] = accounts["community"].apply(hcc_key)
accounts = accounts.merge(hcc_tmp, left_on="community_str", right_on="hcc_id_goal_str", how="left")
print("Goals dibaca dari output lama; tidak ada prediksi ulang.")
print(accounts["account_goal_orientation"].value_counts().to_string())

Goals dibaca dari output lama; tidak ada prediksi ulang.
account_goal_orientation
Insufficient Text           25151
Neutral Engagement            570
Mixed Goals                   327
Promotional / Supportive      217
Critical / Complaint          145
Polarized / Contested          14


## 17. Sentiment Alignment

In [14]:
hcc_dom = hcc_goals.set_index(hcc_goals["hcc_id"].astype(str))["dominant_sentiment"].to_dict()
hcc_valid = hcc_goals.set_index(hcc_goals["hcc_id"].astype(str))["n_valid_text_comments"].to_dict()
VALID_SENT = {"Positive", "Neutral", "Negative"}

def align(row):
    hcc_id = str(row.get("primary_associated_hcc", "") or "")
    if hcc_id == "" or hcc_id not in hcc_dom:
        return "No HCC Association"
    if row["n_valid_text_comments"] < ACCOUNT_GOAL_MIN_COMMENTS or pd.isna(hcc_valid.get(hcc_id)):
        return "Insufficient Text"
    m, h = str(row["dominant_sentiment"]), str(hcc_dom[hcc_id])
    if m in {"", "No text", "nan"} or h in {"", "No text", "nan"}:
        return "Insufficient Text"
    if "Mixed" in m or "Mixed" in h:
        return "Mixed / Inconclusive"
    if m in VALID_SENT and h in VALID_SENT:
        return "Aligned" if m == h else "Not aligned"
    return "Mixed / Inconclusive"

accounts["mass_dominant_sentiment"] = np.where(accounts["is_mass_actor"], accounts["dominant_sentiment"], "")
accounts["associated_hcc_dominant_sentiment"] = accounts["primary_associated_hcc"].fillna("").astype(str).map(hcc_dom).fillna("")
accounts["sentiment_alignment"] = np.where(accounts["is_mass_actor"], accounts.apply(align, axis=1), "")
print(accounts.loc[accounts["is_mass_actor"], "sentiment_alignment"].value_counts().to_string())

sentiment_alignment
Insufficient Text       25151
Aligned                   412
Mixed / Inconclusive      386
Not aligned               257


## 18. Account-Level Three Dimensions Typology

In [15]:
mass_cols = ["username_norm", "shared_hcc_video_count", "n_hcc_accounts_shared_video", "direct_reply_with_hcc_count", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count", "temporal_after_hcc_count", "min_minutes_after_hcc", "median_minutes_after_hcc", "associated_hcc_count", "associated_hcc_ids", "association_basis"]
accounts = accounts.merge(mass_exposure[mass_cols], on="username_norm", how="left")
for c in ["shared_hcc_video_count", "n_hcc_accounts_shared_video", "direct_reply_with_hcc_count", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count", "temporal_after_hcc_count", "associated_hcc_count"]:
    accounts[c] = accounts[c].fillna(0).astype(int)
for c in ["associated_hcc_ids", "association_basis", "individual_subtype", "community_centrality_level", "community_network_descriptor", "mass_network_position", "hcc_goal_orientation", "hcc_goal_confidence"]:
    accounts[c] = accounts[c].fillna("")

account_cols = [
    "username", "actor_type_primary", "is_individual_actor", "individual_subtype", "is_hcc_member", "is_lcn_member", "is_mass_actor", "individual_in_hcc",
    "community", "degree", "weighted_degree", "betweenness", "community_centrality_level", "community_network_descriptor", "mass_network_position",
    "shared_hcc_video_count", "n_hcc_accounts_shared_video", "direct_reply_with_hcc_count", "mass_replies_to_hcc_count", "hcc_replies_to_mass_count",
    "temporal_after_hcc_count", "min_minutes_after_hcc", "median_minutes_after_hcc", "associated_hcc_count", "associated_hcc_ids",
    "primary_associated_hcc", "association_basis", "mass_exposure_level", "mass_exposure_score", "mass_observed_target_brand", "associated_hcc_brand_context",
    "target_brand_primary", "target_brand_label", "target_scope", "target_product_category", "target_video_count", "target_thread_count", "target_confidence", "target_source",
    "n_comments", "n_valid_text_comments", "dominant_sentiment", "positive_ratio", "neutral_ratio", "negative_ratio", "avg_sentiment_confidence",
    "account_goal_orientation", "account_goal_confidence", "hcc_goal_orientation", "hcc_goal_confidence", "mass_dominant_sentiment", "associated_hcc_dominant_sentiment", "sentiment_alignment",
]
for c in account_cols:
    if c not in accounts.columns:
        accounts[c] = ""
account_actor_type = accounts[account_cols].copy()
account_actor_type.to_csv(TABLE_DIR / "account_actor_type.csv", index=False)

accounts["actor_subtype_or_position"] = np.select(
    [accounts["actor_type_primary"].eq("Individual Actor"), accounts["actor_type_primary"].eq("Community Actor"), accounts["actor_type_primary"].eq("Mass Actor")],
    [accounts["individual_subtype"].replace("", "Verified Individual Source"), accounts["community_network_descriptor"].replace("", "Community Actor"), accounts["mass_network_position"]],
    default="",
)
three_dimensions_typology_account = accounts[["username", "actor_type_primary", "actor_subtype_or_position", "target_brand_primary", "target_scope", "target_product_category", "account_goal_orientation", "dominant_sentiment", "primary_associated_hcc", "mass_exposure_level", "sentiment_alignment"]].copy()
three_dimensions_typology_account.to_csv(TABLE_DIR / "three_dimensions_typology_account.csv", index=False)
mass_exposure.to_csv(TABLE_DIR / "mass_actor_exposure.csv", index=False)
print(f"account_actor_type rows: {len(account_actor_type):,}")

account_actor_type rows: 26,424


## 19. HCC-Level Three Dimensions Typology

In [16]:
hcc_accounts = accounts[accounts["is_hcc_member"]].copy()
community_actor_summary = hcc_accounts.groupby("community_str", as_index=False).agg(
    hcc_id=("community_str", "first"), community_size=("username", "size"),
    n_verified_individuals_in_hcc=("individual_in_hcc", "sum"),
    mean_degree=("degree", "mean"), mean_weighted_degree=("weighted_degree", "mean"), mean_betweenness=("betweenness", "mean"),
    primary_brand=("primary_brand", lambda s: s.dropna().astype(str).mode().iloc[0] if len(s.dropna()) else "Not identified"),
    brand_label_auto=("brand_label_auto", lambda s: s.dropna().astype(str).mode().iloc[0] if len(s.dropna()) else "Not identified"),
).drop(columns=["community_str"])
community_actor_summary["target_scope"] = community_actor_summary["brand_label_auto"].apply(target_scope)
community_actor_summary = community_actor_summary.merge(
    hcc_tmp[["hcc_id_goal_str", "hcc_goal_orientation", "hcc_goal_confidence", "hcc_positive_ratio", "hcc_neutral_ratio", "hcc_negative_ratio", "hcc_dominant_sentiment", "hcc_n_comments"]],
    left_on="hcc_id", right_on="hcc_id_goal_str", how="left"
).drop(columns=["hcc_id_goal_str"])

primary_mass = accounts[accounts["is_mass_actor"] & accounts["primary_associated_hcc"].fillna("").ne("")].copy()
mass_by_hcc = primary_mass.groupby("primary_associated_hcc").agg(
    n_mass_actors_associated=("username", "nunique"),
    n_mass_direct_interaction=("mass_exposure_level", lambda s: int((s == "Direct Interaction").sum())),
    n_mass_temporal_exposure=("mass_exposure_level", lambda s: int((s == "Temporal-Context Exposure").sum())),
    n_mass_shared_video_exposure=("mass_exposure_level", lambda s: int((s == "Shared-Video Exposure").sum())),
    dominant_mass_exposure_level=("mass_exposure_level", lambda s: s.value_counts().index[0] if len(s) else ""),
).reset_index().rename(columns={"primary_associated_hcc": "hcc_id"})
align_rate = primary_mass.groupby("primary_associated_hcc")["sentiment_alignment"].apply(lambda s: s.eq("Aligned").sum() / max(1, s.isin(["Aligned", "Not aligned"]).sum()) if s.isin(["Aligned", "Not aligned"]).sum() else np.nan).reset_index().rename(columns={"primary_associated_hcc": "hcc_id", "sentiment_alignment": "mass_sentiment_alignment_rate"})
community_actor_summary = community_actor_summary.merge(mass_by_hcc, on="hcc_id", how="left").merge(align_rate, on="hcc_id", how="left")
for c in ["n_mass_actors_associated", "n_mass_direct_interaction", "n_mass_temporal_exposure", "n_mass_shared_video_exposure"]:
    community_actor_summary[c] = community_actor_summary[c].fillna(0).astype(int)
community_actor_summary["dominant_mass_exposure_level"] = community_actor_summary["dominant_mass_exposure_level"].fillna("No Observed HCC Exposure")
community_actor_summary = community_actor_summary.rename(columns={"hcc_positive_ratio": "positive_ratio", "hcc_neutral_ratio": "neutral_ratio", "hcc_negative_ratio": "negative_ratio"})
community_actor_summary.to_csv(TABLE_DIR / "community_actor_summary.csv", index=False)

def interp(r):
    return (f"HCC {r['hcc_id']} merupakan Community Actor dengan konteks target {r['primary_brand']} dan orientasi pesan {r['hcc_goal_orientation']}. "
            f"HCC ini memiliki {int(r['n_mass_actors_associated'])} Mass Actor yang menunjukkan paparan teramati pada video atau percakapan yang sama. "
            "Temuan tersebut menunjukkan asosiasi struktural dan naratif, bukan bukti kausal pengaruh atau niat manipulatif.")

three_dimensions_typology_hcc = community_actor_summary.copy()
three_dimensions_typology_hcc["actor_type"] = "Community Actor"
three_dimensions_typology_hcc["dominant_sentiment"] = three_dimensions_typology_hcc["hcc_dominant_sentiment"]
three_dimensions_typology_hcc["n_associated_mass_actors"] = three_dimensions_typology_hcc["n_mass_actors_associated"]
three_dimensions_typology_hcc["interpretation_safe"] = three_dimensions_typology_hcc.apply(interp, axis=1)
three_dimensions_typology_hcc = three_dimensions_typology_hcc[["hcc_id", "actor_type", "community_size", "primary_brand", "brand_label_auto", "target_scope", "hcc_goal_orientation", "hcc_goal_confidence", "dominant_sentiment", "n_associated_mass_actors", "dominant_mass_exposure_level", "mass_sentiment_alignment_rate", "interpretation_safe"]]
three_dimensions_typology_hcc.to_csv(TABLE_DIR / "three_dimensions_typology_hcc.csv", index=False)
print(f"community_actor_summary rows: {len(community_actor_summary):,}")

community_actor_summary rows: 42


## 20. Summary Tables

In [17]:
individual_actor_summary = (
    accounts[accounts["actor_type_primary"].eq("Individual Actor")]
    .groupby("individual_subtype", dropna=False)
    .agg(n_accounts=("username", "nunique"), n_hcc_members=("is_hcc_member", "sum"), n_lcn_members=("is_lcn_member", "sum"), n_comments=("n_comments", "sum"))
    .reset_index()
    if accounts["actor_type_primary"].eq("Individual Actor").any()
    else pd.DataFrame(columns=["individual_subtype", "n_accounts", "n_hcc_members", "n_lcn_members", "n_comments"])
)
individual_actor_summary.to_csv(TABLE_DIR / "individual_actor_summary.csv", index=False)

summary_rows = []
total_accounts, total_comments = len(accounts), accounts["n_comments"].sum()
for actor_type in ACTOR_TYPES:
    g = accounts[accounts["actor_type_primary"].eq(actor_type)]
    pos, neu, neg = g["positive_count"].sum(), g["neutral_count"].sum(), g["negative_count"].sum()
    valid = pos + neu + neg
    summary_rows.append({
        "actor_type_primary": actor_type, "n_accounts": len(g), "account_percentage": len(g) / max(1, total_accounts),
        "n_comments": int(g["n_comments"].sum()), "comment_percentage": g["n_comments"].sum() / max(1, total_comments),
        "positive_ratio": pos / valid if valid else 0, "neutral_ratio": neu / valid if valid else 0, "negative_ratio": neg / valid if valid else 0,
        "dominant_goal_orientation": g["account_goal_orientation"].value_counts().index[0] if len(g) else "",
    })
actor_type_summary = pd.DataFrame(summary_rows)
actor_type_summary.to_csv(TABLE_DIR / "actor_type_summary.csv", index=False)
accounts.groupby(["actor_type_primary", "target_scope", "target_brand_primary"], as_index=False).agg(n_accounts=("username", "nunique"), n_comments=("n_comments", "sum")).to_csv(TABLE_DIR / "actor_type_by_target.csv", index=False)
accounts.groupby(["actor_type_primary", "account_goal_orientation"], as_index=False).agg(n_accounts=("username", "nunique"), n_comments=("n_comments", "sum")).to_csv(TABLE_DIR / "actor_type_by_goals.csv", index=False)

method_thresholds = pd.DataFrame([
    ["MASS_TEMPORAL_WINDOW_MINUTES", MASS_TEMPORAL_WINDOW_MINUTES, "Same-video Mass comment after HCC comment within window."],
    ["CENTRALITY_Q25", CENTRALITY_Q25, "Low centrality percentile within each HCC."],
    ["CENTRALITY_Q75", CENTRALITY_Q75, "High centrality percentile within each HCC."],
    ["ACCOUNT_GOAL_MIN_COMMENTS", ACCOUNT_GOAL_MIN_COMMENTS, "Minimum valid account comments."],
    ["GOAL_POSITIVE_THRESHOLD", GOAL_POSITIVE_THRESHOLD, "Promotional / Supportive threshold."],
    ["GOAL_NEGATIVE_THRESHOLD", GOAL_NEGATIVE_THRESHOLD, "Critical / Complaint threshold."],
    ["GOAL_NEUTRAL_THRESHOLD", GOAL_NEUTRAL_THRESHOLD, "Neutral Engagement threshold."],
    ["GOAL_POLARIZED_THRESHOLD", GOAL_POLARIZED_THRESHOLD, "Polarized / Contested threshold."],
    ["EXPOSURE_SCORE_WEIGHTS", str(EXPOSURE_WEIGHTS), "Descriptive score weights, not causal influence."],
    ["PRIMARY_HCC_TIE_BREAK", "direct desc > temporal desc > shared-video desc > smallest HCC ID", "Deterministic primary association rule."],
    ["REGISTRY_VERIFICATION_RULE", "verification_status == Verified", "Candidate rows are never Individual Actor."],
], columns=["parameter", "value", "note"])
method_thresholds.to_csv(TABLE_DIR / "method_thresholds.csv", index=False)
actor_type_summary

,actor_type_primary,n_accounts,account_percentage,n_comments,comment_percentage,positive_ratio,neutral_ratio,negative_ratio,dominant_goal_orientation
0,Individual Actor,0,0.00000,0,0.000000,0.000000,0.000000,0.000000,
1,Community Actor,218,0.00825,1009,0.029811,0.444995,0.406343,0.148662,Neutral Engagement
2,Mass Actor,26206,0.99175,32838,0.970189,0.262930,0.522264,0.214806,Insufficient Text


## 21. Visualisasi

In [18]:
visual_files = []
def save(path):
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    visual_files.append(path)
    print(f"Disimpan: {path}")

fig, ax = plt.subplots(figsize=(8, 4.5))
s = accounts["actor_type_primary"].value_counts().reindex(ACTOR_TYPES, fill_value=0)
ax.bar(s.index, s.values, color=["#4C72B0", "#55A868", "#C9A227"])
ax.set_title("Actor Type Distribution — Accounts"); ax.set_xlabel("Actor type"); ax.set_ylabel("Number of accounts"); ax.tick_params(axis="x", rotation=15)
for i, v in enumerate(s.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
save(VIZ_DIR / "actor_type_distribution_accounts.png")

fig, ax = plt.subplots(figsize=(8, 4.5))
s = accounts.groupby("actor_type_primary")["n_comments"].sum().reindex(ACTOR_TYPES, fill_value=0)
ax.bar(s.index, s.values, color=["#4C72B0", "#55A868", "#C9A227"])
ax.set_title("Actor Type Distribution — Comments"); ax.set_xlabel("Actor type"); ax.set_ylabel("Number of comments"); ax.tick_params(axis="x", rotation=15)
for i, v in enumerate(s.values): ax.text(i, v, f"{int(v):,}", ha="center", va="bottom", fontsize=9)
save(VIZ_DIR / "actor_type_distribution_comments.png")

def stacked(col, title, path):
    ct = pd.crosstab(accounts["actor_type_primary"], accounts[col]).reindex(index=ACTOR_TYPES, fill_value=0)
    fig, ax = plt.subplots(figsize=(10, 5))
    bottom = np.zeros(len(ct)); colors = plt.cm.Set2(np.linspace(0, 1, max(3, len(ct.columns))))
    for color, c in zip(colors, ct.columns):
        ax.bar(ct.index, ct[c], bottom=bottom, label=c, color=color); bottom += ct[c].values
    ax.set_title(title); ax.set_xlabel("Actor type"); ax.set_ylabel("Number of accounts"); ax.tick_params(axis="x", rotation=15)
    ax.legend(loc="center left", bbox_to_anchor=(1, .5), fontsize=8); save(path)
stacked("account_goal_orientation", "Actor Type by Goal Orientation", VIZ_DIR / "actor_type_by_goal_orientation.png")
stacked("target_scope", "Actor Type by Target Scope", VIZ_DIR / "actor_type_by_target_scope.png")

sent = accounts.groupby("actor_type_primary")[["positive_count", "neutral_count", "negative_count"]].sum().reindex(ACTOR_TYPES, fill_value=0)
sent = sent.div(sent.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
fig, ax = plt.subplots(figsize=(8, 5)); bottom = np.zeros(len(sent))
for c, label, color in [("positive_count", "Positive", "#55A868"), ("neutral_count", "Neutral", "#8C8C8C"), ("negative_count", "Negative", "#C44E52")]:
    ax.bar(sent.index, sent[c], bottom=bottom, label=label, color=color); bottom += sent[c].values
ax.set_title("Sentiment Distribution by Actor Type"); ax.set_xlabel("Actor type"); ax.set_ylabel("Share of valid comments"); ax.tick_params(axis="x", rotation=15); ax.legend()
save(VIZ_DIR / "sentiment_distribution_by_actor_type.png")

fig, ax = plt.subplots(figsize=(9, 4.8))
s = mass_exposure["mass_exposure_level"].value_counts().reindex(EXPOSURE_ORDER, fill_value=0)
ax.bar(s.index, s.values, color="#4C72B0"); ax.set_title("Mass Actor Observed Exposure Level Distribution"); ax.set_xlabel("Observed exposure level"); ax.set_ylabel("Number of Mass Actors"); ax.tick_params(axis="x", rotation=20)
for i, v in enumerate(s.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
save(VIZ_DIR / "mass_exposure_level_distribution.png")

fig, ax = plt.subplots(figsize=(10, 5))
ct = pd.crosstab(accounts.loc[accounts["is_mass_actor"], "mass_exposure_level"], accounts.loc[accounts["is_mass_actor"], "dominant_sentiment"]).reindex(index=EXPOSURE_ORDER, fill_value=0)
bottom = np.zeros(len(ct))
for sent_label, color in [("Positive", "#55A868"), ("Neutral", "#8C8C8C"), ("Negative", "#C44E52"), ("Mixed", "#8172B2"), ("No text", "#BBBBBB")]:
    if sent_label in ct.columns:
        ax.bar(ct.index, ct[sent_label], bottom=bottom, label=sent_label, color=color); bottom += ct[sent_label].values
ax.set_title("Mass Observed Exposure by Dominant Sentiment"); ax.set_xlabel("Observed exposure level"); ax.set_ylabel("Number of Mass Actors"); ax.tick_params(axis="x", rotation=20); ax.legend(loc="center left", bbox_to_anchor=(1, .5), fontsize=8)
save(VIZ_DIR / "mass_exposure_sentiment_distribution.png")

fig, ax = plt.subplots(figsize=(10, 6))
top = primary_mass["primary_associated_hcc"].value_counts().head(20).sort_values()
ax.barh([f"HCC {x}" for x in top.index], top.values, color="#55A868"); ax.set_title("Top HCC by Associated Mass Actors — Observed Association"); ax.set_xlabel("Number of associated Mass Actors"); ax.set_ylabel("HCC")
save(VIZ_DIR / "mass_association_by_hcc.png")

fig, ax = plt.subplots(figsize=(8, 4.8))
s = accounts.loc[accounts["is_mass_actor"], "sentiment_alignment"].value_counts()
ax.bar(s.index, s.values, color="#8172B2"); ax.set_title("Community-Mass Sentiment Alignment"); ax.set_xlabel("Sentiment alignment"); ax.set_ylabel("Number of Mass Actors"); ax.tick_params(axis="x", rotation=20)
for i, v in enumerate(s.values): ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
save(VIZ_DIR / "community_mass_sentiment_alignment.png")

piv = pd.crosstab(accounts["actor_type_primary"], accounts["target_scope"]).reindex(index=ACTOR_TYPES, fill_value=0)
fig, ax = plt.subplots(figsize=(max(8, len(piv.columns) * 2.1), 4.8))
im = ax.imshow(piv.values, cmap="YlGnBu", aspect="auto")
ax.set_title("Three Dimensions Heatmap — Actor Type, Target Scope, and Dominant Goal")
ax.set_xlabel("Target scope"); ax.set_ylabel("Actor type")
ax.set_xticks(np.arange(len(piv.columns))); ax.set_xticklabels(piv.columns, rotation=20, ha="right")
ax.set_yticks(np.arange(len(ACTOR_TYPES))); ax.set_yticklabels(ACTOR_TYPES)
for i, a in enumerate(ACTOR_TYPES):
    for j, t in enumerate(piv.columns):
        sub = accounts[(accounts["actor_type_primary"] == a) & (accounts["target_scope"] == t)]
        goal = sub["account_goal_orientation"].value_counts().index[0] if len(sub) else ""
        ax.text(j, i, f"{len(sub):,}\n{goal[:18]}" if len(sub) else "0", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Number of accounts")
save(VIZ_DIR / "three_dimensions_actor_target_goal_heatmap.png")
print(f"Visualisasi PNG dibuat: {len(visual_files)}")

Disimpan: output\rm2_actor_type\visualisasi\actor_type_distribution_accounts.png
Disimpan: output\rm2_actor_type\visualisasi\actor_type_distribution_comments.png


Disimpan: output\rm2_actor_type\visualisasi\actor_type_by_goal_orientation.png
Disimpan: output\rm2_actor_type\visualisasi\actor_type_by_target_scope.png


Disimpan: output\rm2_actor_type\visualisasi\sentiment_distribution_by_actor_type.png
Disimpan: output\rm2_actor_type\visualisasi\mass_exposure_level_distribution.png


Disimpan: output\rm2_actor_type\visualisasi\mass_exposure_sentiment_distribution.png


Disimpan: output\rm2_actor_type\visualisasi\mass_association_by_hcc.png
Disimpan: output\rm2_actor_type\visualisasi\community_mass_sentiment_alignment.png


Disimpan: output\rm2_actor_type\visualisasi\three_dimensions_actor_target_goal_heatmap.png
Visualisasi PNG dibuat: 10


## 22. Gephi Export

In [19]:
gephi_hcc_nodes_actor = gephi_sentiment_nodes.copy() if not gephi_sentiment_nodes.empty else hcc_nodes.copy()
gephi_hcc_nodes_actor["id_norm"] = gephi_hcc_nodes_actor["id"].apply(normalize_username)
gephi_hcc_nodes_actor = gephi_hcc_nodes_actor.merge(
    accounts[["username_norm", "actor_type_primary", "individual_subtype", "community_centrality_level", "community_network_descriptor", "target_brand_primary", "target_scope", "account_goal_orientation", "hcc_goal_orientation"]],
    left_on="id_norm", right_on="username_norm", how="left",
).drop(columns=["id_norm", "username_norm"])
gephi_hcc_nodes_actor.to_csv(GEPHI_DIR / "gephi_hcc_nodes_actor_type.csv", index=False)
shutil.copyfile(HCC_EDGES_PATH, GEPHI_DIR / "gephi_hcc_edges_actor_type.csv")

agg_nodes, agg_edges = [], []
for _, r in community_actor_summary.iterrows():
    h = str(r["hcc_id"])
    agg_nodes.append({"id": f"HCC_{h}", "label": f"HCC {h}", "actor_type_primary": "Community Actor", "actor_subtype": "HCC final community", "associated_hcc": h, "target_brand": r["primary_brand"], "goal_orientation": r["hcc_goal_orientation"], "account_count": int(r["community_size"]), "comment_count": int(r["hcc_n_comments"]) if pd.notna(r.get("hcc_n_comments", np.nan)) else 0, "exposure_level": ""})

for _, r in accounts[accounts["actor_type_primary"].eq("Individual Actor")].iterrows():
    node_id = "IND_" + re.sub(r"[^A-Za-z0-9_]+", "_", r["username_norm"])
    agg_nodes.append({"id": node_id, "label": r["username"], "actor_type_primary": "Individual Actor", "actor_subtype": r["individual_subtype"], "associated_hcc": r["community_str"] if r["is_hcc_member"] else "", "target_brand": r["target_brand_primary"], "goal_orientation": r["account_goal_orientation"], "account_count": 1, "comment_count": int(r["n_comments"]), "exposure_level": ""})

mass_segments = primary_mass.groupby(["primary_associated_hcc", "mass_exposure_level", "dominant_sentiment"], dropna=False).agg(account_count=("username", "nunique"), comment_count=("n_comments", "sum"), goal_orientation=("account_goal_orientation", lambda s: s.value_counts().index[0] if len(s) else "")).reset_index()
for _, r in mass_segments.iterrows():
    h, exp, sent = str(r["primary_associated_hcc"]), str(r["mass_exposure_level"]), str(r["dominant_sentiment"])
    node_id = "MASS_HCC_" + re.sub(r"[^A-Za-z0-9_]+", "_", f"{h}_{exp}_{sent}")
    target_brand = hcc_brand_map.loc[h, "primary_brand"] if h in hcc_brand_map.index else "Not identified"
    agg_nodes.append({"id": node_id, "label": f"Mass | HCC {h} | {exp} | {sent}", "actor_type_primary": "Mass Actor", "actor_subtype": "Aggregated Mass segment", "associated_hcc": h, "target_brand": target_brand, "goal_orientation": r["goal_orientation"], "account_count": int(r["account_count"]), "comment_count": int(r["comment_count"]), "exposure_level": exp})
    seg = primary_mass[(primary_mass["primary_associated_hcc"].astype(str) == h) & (primary_mass["mass_exposure_level"].astype(str) == exp) & (primary_mass["dominant_sentiment"].astype(str) == sent)]
    agg_edges.append({"source": f"HCC_{h}", "target": node_id, "weight": int(r["account_count"]), "edge_type": "Observed Community-Mass Association", "shared_video_count": int(seg["shared_hcc_video_count"].sum()), "direct_interaction_count": int(seg["direct_reply_with_hcc_count"].sum())})

gephi_three_actor_aggregate_nodes = pd.DataFrame(agg_nodes).drop_duplicates("id")
gephi_three_actor_aggregate_edges = pd.DataFrame(agg_edges)
if gephi_three_actor_aggregate_edges.empty:
    gephi_three_actor_aggregate_edges = pd.DataFrame(columns=["source", "target", "weight", "edge_type", "shared_video_count", "direct_interaction_count"])
else:
    gephi_three_actor_aggregate_edges = gephi_three_actor_aggregate_edges.groupby(["source", "target", "edge_type"], as_index=False).agg(weight=("weight", "sum"), shared_video_count=("shared_video_count", "sum"), direct_interaction_count=("direct_interaction_count", "sum"))
gephi_three_actor_aggregate_nodes.to_csv(GEPHI_DIR / "gephi_three_actor_aggregate_nodes.csv", index=False)
gephi_three_actor_aggregate_edges.to_csv(GEPHI_DIR / "gephi_three_actor_aggregate_edges.csv", index=False)
print("Gephi export dibuat:")
for name in ["gephi_hcc_nodes_actor_type.csv", "gephi_hcc_edges_actor_type.csv", "gephi_three_actor_aggregate_nodes.csv", "gephi_three_actor_aggregate_edges.csv"]:
    print(f"- {GEPHI_DIR / name}")

Gephi export dibuat:
- output\rm2_actor_type\gephi\gephi_hcc_nodes_actor_type.csv
- output\rm2_actor_type\gephi\gephi_hcc_edges_actor_type.csv
- output\rm2_actor_type\gephi\gephi_three_actor_aggregate_nodes.csv
- output\rm2_actor_type\gephi\gephi_three_actor_aggregate_edges.csv


## 23. Interpretation Guide

Interpretasi harus dibaca sebagai tipologi deskriptif tiga dimensi: actor type, target, dan goals. Target
menunjukkan konteks brand/video atau kategori produk yang teramati, bukan korban atau sasaran serangan.
Observed Community-Mass exposure menunjukkan interaksi langsung, kedekatan temporal, atau kesamaan ruang video,
bukan pengaruh kausal.

## 24. Limitasi

1. Individual Actor hanya dapat diklasifikasikan melalui registry manual dengan `verification_status = "Verified"`.
2. Metadata kreator terbatas pada kolom yang tersedia; follower count dan status verifikasi platform tidak tersedia.
3. Parent reply dihitung hanya ketika `parent_user` valid dan dapat dipetakan.
4. Temporal exposure terbatas pada jendela 60 menit setelah komentar HCC pada video yang sama.
5. Sentiment alignment bukan bukti perubahan opini.
6. LCN membership untuk Mass Actor adalah posisi jaringan sekunder; hanya HCC final yang menjadi Community Actor.

## 25. Final Validation Report

In [20]:
errors = []
if accounts["actor_type_primary"].isna().any():
    errors.append("Ada akun tanpa actor_type_primary")
if set(accounts["actor_type_primary"].unique()) - set(ACTOR_TYPES):
    errors.append("Ada actor type di luar tiga kategori")
if len(accounts) != accounts["username_norm"].nunique():
    errors.append("Jumlah account rows tidak sama dengan username unik")
if verified_set and not accounts.loc[accounts["username_norm"].isin(verified_set), "actor_type_primary"].eq("Individual Actor").all():
    errors.append("Tidak semua Verified registry menjadi Individual Actor")
if len(accounts[accounts["username_norm"].isin(set(hcc_nodes["id_norm"])) & accounts["actor_type_primary"].eq("Mass Actor")]):
    errors.append("Ada HCC member menjadi Mass Actor")
candidate_set = set(individual_actor_candidates["username"].apply(normalize_username)) if len(individual_actor_candidates) else set()
if len(accounts[accounts["username_norm"].isin(candidate_set - verified_set) & accounts["actor_type_primary"].eq("Individual Actor")]):
    errors.append("Candidate otomatis menjadi Individual Actor")
if hcc_nodes["id_norm"].duplicated().any():
    errors.append("Duplicate HCC username setelah normalisasi")
if temporal_records:
    td = pd.DataFrame(temporal_records)
    if (td["minutes_after_hcc"] < 0).any() or (td["minutes_after_hcc"] > MASS_TEMPORAL_WINDOW_MINUTES).any():
        errors.append("Temporal exposure di luar window")
if len(hcc_nodes) != 218 or hcc_nodes["community"].nunique() != 42 or len(hcc_edges) != 464:
    errors.append("Baseline HCC bukan 42/218/464")

output_files = sorted([p for p in OUT_DIR.rglob("*") if p.is_file()])
print("=" * 88)
print("FINAL VALIDATION REPORT — RM2 ACTOR TYPE TYPOLOGY")
print("=" * 88)
print(f"Commit HEAD sebelum implementasi : {HEAD_COMMIT_BEFORE_IMPLEMENTATION}")
print(f"Total akun                       : {len(accounts):,}")
for actor_type in ACTOR_TYPES:
    print(f"{actor_type:<32s}: {(accounts['actor_type_primary'] == actor_type).sum():,}")
print(f"Verified Individual Actor         : {len(verified_registry):,}")
print(f"Verified individual dalam HCC     : {accounts['individual_in_hcc'].sum():,}")
print(f"LCN Non-HCC                       : {((accounts['is_mass_actor']) & (accounts['mass_network_position'] == 'LCN Non-HCC')).sum():,}")
print(f"Outside LCN                       : {((accounts['is_mass_actor']) & (accounts['mass_network_position'] == 'Outside LCN')).sum():,}")
print(f"Mass Direct Interaction           : {(mass_exposure['mass_exposure_level'] == 'Direct Interaction').sum():,}")
print(f"Mass Temporal-Context Exposure    : {(mass_exposure['mass_exposure_level'] == 'Temporal-Context Exposure').sum():,}")
print(f"Mass Shared-Video Exposure        : {(mass_exposure['mass_exposure_level'] == 'Shared-Video Exposure').sum():,}")
print(f"Mass No Observed HCC Exposure     : {(mass_exposure['mass_exposure_level'] == 'No Observed HCC Exposure').sum():,}")
print("\nDistribusi target_scope:")
print(accounts["target_scope"].value_counts().to_string())
print("\nDistribusi goals per actor type:")
print(pd.crosstab(accounts["actor_type_primary"], accounts["account_goal_orientation"]).to_string())
print("\nSentiment alignment Mass Actor:")
print(accounts.loc[accounts["is_mass_actor"], "sentiment_alignment"].value_counts().to_string())
print(f"\nJumlah file output actor type     : {len(output_files):,}")
for p in output_files:
    print(f"- {p}")
if errors:
    print("\nVALIDATION ERRORS:")
    for e in errors:
        print(f"- {e}")
    raise AssertionError("Validasi gagal")
print("\nVALIDASI OK")
print("Notebook ini tidak menulis output RM1 atau output RM2 sentiment; output baru hanya di output/rm2_actor_type/ dan config registry.")

FINAL VALIDATION REPORT — RM2 ACTOR TYPE TYPOLOGY
Commit HEAD sebelum implementasi : 4fcbdc93788f990e3eb51d2f25b57b09eb66ec27
Total akun                       : 26,424
Individual Actor                : 0
Community Actor                 : 218
Mass Actor                      : 26,206
Verified Individual Actor         : 0
Verified individual dalam HCC     : 0
LCN Non-HCC                       : 506
Outside LCN                       : 25,700
Mass Direct Interaction           : 28
Mass Temporal-Context Exposure    : 2,294
Mass Shared-Video Exposure        : 23,884
Mass No Observed HCC Exposure     : 0

Distribusi target_scope:
target_scope
Single-brand Context          25838
Cross-brand Context             580
Unidentified Brand Context        6

Distribusi goals per actor type:
account_goal_orientation  Critical / Complaint  Insufficient Text  Mixed Goals  Neutral Engagement  Polarized / Contested  Promotional / Supportive
actor_type_primary                                                 